<a href="https://colab.research.google.com/github/Joe-Something/AAI2026/blob/main/Ex1_CustomerSupport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# ==============================================================================
# Exercise 1: Prompt Chaining for Customer Support AI
# Environment: Google Colab
# Library: google-genai
# ==============================================================================

!pip install -q google-genai

import os
import time
from google import genai
from google.genai import errors
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

PRIMARY_MODEL = "gemini-3.6-flash"

def safe_generate_content(prompt, system_instruction=None):
    config = None
    if system_instruction:
        config = genai.types.GenerateContentConfig(system_instruction=system_instruction)

    for attempt in range(5):
        try:
            return client.models.generate_content(
                model=PRIMARY_MODEL,
                contents=prompt,
                config=config
            )
        except errors.ServerError:
            # Server errors (500/503) exponential backoff
            time.sleep(2 ** attempt)
        except errors.APIError as e:
            if e.code == 429:
                # Rate limit hit (429) - wait 20-30s before retrying
                wait_time = 20 + (attempt * 5)
                print(f"[Rate Limit 429] Waiting {wait_time}s before retry (attempt {attempt + 1}/5)...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"API Error ({e.code}): {e.message}")
        except Exception as e:
            # Handle ClientError status 429 directly if thrown as generic ClientError
            if getattr(e, 'code', None) == 429 or "429" in str(e):
                wait_time = 20 + (attempt * 5)
                print(f"[Rate Limit 429] Waiting {wait_time}s before retry (attempt {attempt + 1}/5)...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"Unexpected error: {e}")

    raise RuntimeError(f"Failed to generate content with model {PRIMARY_MODEL} after retries.")

customer_email = """
Hi Support,
I was double charged on my invoice #INV-98234 for $149.00 last Tuesday.
I tried calling customer service but was placed on hold for 45 minutes before being disconnected.
This is completely unacceptable and I need this refunded immediately or I will file a chargeback with my bank!
- Alex Johnson
"""

print("=== RAW CUSTOMER EMAIL ===")
print(customer_email.strip())
print("=" * 50)

# ------------------------------------------------------------------------------
# STEP 1: Classification & Priority Assessment
# ------------------------------------------------------------------------------
step1_prompt = f"""
You are an incoming email classifier for customer support.
Analyze the following customer email and extract:
1. Category (Choose one: Billing, Technical Support, Returns, General Inquiry)
2. Priority (Choose one: High, Medium, Low)
3. Main Concern (1 sentence summary)

Email:
{customer_email}

Respond in the following format:
Category: <Category>
Priority: <Priority>
Summary: <Summary>
"""

response_step1 = safe_generate_content(step1_prompt)
step1_output = response_step1.text

print("\n--- STEP 1 OUTPUT: CLASSIFICATION ---")
print(step1_output.strip())

# Pause to avoid exceeding 5 Requests Per Minute (RPM) free tier quota
time.sleep(12)

# ------------------------------------------------------------------------------
# STEP 2: Information Extraction & Escalation Logic
# ------------------------------------------------------------------------------
step2_prompt = f"""
You are an operations triage specialist. Review the customer email and the Step 1 classification.

Customer Email:
{customer_email}

Step 1 Classification:
{step1_output}

Task:
1. List any missing information required to resolve the issue (e.g., account number, transaction ID).
2. Determine if Immediate Escalation to a Human Manager is required (YES/NO).
   Rules for Escalation = YES: Priority is High AND mentions phone disconnections, legal threats, or chargebacks.
3. Provide a brief 1-sentence reasoning for the escalation flag.

Respond in format:
Missing Info: <details or None>
Human Escalation Required: <YES/NO>
Escalation Reason: <reason>
"""

response_step2 = safe_generate_content(step2_prompt)
step2_output = response_step2.text

print("\n--- STEP 2 OUTPUT: EXTRACTION & TRIAGE ---")
print(step2_output.strip())

# Pause to respect rate limits
time.sleep(12)

# ------------------------------------------------------------------------------
# STEP 3: Response Generation with Strict Prompt Constraints
# ------------------------------------------------------------------------------
step3_prompt = f"""
You are a senior customer support representative.
Draft a response email using the context provided below.

Context:
Customer Email: {customer_email}
Classification: {step1_output}
Triage Assessment: {step2_output}

Response Constraints:
- Tone: Highly empathetic, calm, and professional.
- Acknowledge the phone disconnection frustration specifically.
- If Escalation is YES, assure them the issue has been flagged to a senior specialist for urgent refund processing.
- Request any missing information identified in Step 2 if applicable.
- DO NOT use generic phrases like "Dear Valued Customer" or "We apologize for any inconvenience caused."
- Keep the length under 150 words.
"""

response_step3 = safe_generate_content(step3_prompt)
step3_output = response_step3.text

print("\n--- STEP 3 OUTPUT: FINAL RESPONSE ---")
print(step3_output.strip())

=== RAW CUSTOMER EMAIL ===
Hi Support,
I was double charged on my invoice #INV-98234 for $149.00 last Tuesday.
I tried calling customer service but was placed on hold for 45 minutes before being disconnected.
This is completely unacceptable and I need this refunded immediately or I will file a chargeback with my bank!
- Alex Johnson

--- STEP 1 OUTPUT: CLASSIFICATION ---
Category: Billing
Priority: High
Summary: The customer was double charged $149.00 on invoice #INV-98234 and is requesting an immediate refund after being unable to reach support by phone.

--- STEP 2 OUTPUT: EXTRACTION & TRIAGE ---
Missing Info: Account ID/Number, payment method details, or bank transaction ID for the duplicate charge
Human Escalation Required: YES
Escalation Reason: The ticket priority is High and the customer explicitly mentions phone disconnections and a threat to file a chargeback.
[Rate Limit 429] Waiting 25s before retry (attempt 2/5)...

--- STEP 3 OUTPUT: FINAL RESPONSE ---
Hi Alex,

I am deepl